# Notebook A: Q1 analysis (induction heads as an early warning)
 

## Setup 1: connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('collapse found:', os.path.exists('/content/drive/MyDrive/nlp_p1/collapse/gen0_pool/config.json'))
os.makedirs('/content/drive/MyDrive/nlp_p1/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/nlp_p1/collapse_ext', exist_ok=True)
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "no GPU (fine for analysis, slower)"


## Setup 2: Python 3.10 with the project's versions

In [ ]:
!pip -q install uv
!uv venv /content/py310 --python 3.10 -q --allow-existing
!uv pip install -q --python /content/py310/bin/python "jax[cuda12]==0.4.26" "jaxlib==0.4.26" "equinox==0.11.10" "optax==0.2.2" "nptyping==2.5.0" "numpy==1.26.4" "scipy==1.15.3" "h5py==3.16.0" "matplotlib==3.10.9" "tqdm==4.70.1"
!XLA_PYTHON_CLIENT_PREALLOCATE=false /content/py310/bin/python -c "import jax; print(jax.__version__, jax.devices())"


## Setup 3: the code 

In [ ]:
BRANCH = 'mivuyo/q1-induction-heads'   # branch name
%cd /content
!rm -rf icl-dynamics && git clone -q -b $BRANCH https://github.com/Bigjay101/icl-dynamics.git
%cd /content/icl-dynamics
!git log --oneline -1

## Setup 4: links and settings 
This makes `collapse/`, `collapse_ext/` and `results/` inside the code folder point to your Drive, so every command in the guide works unchanged and every output lands in Drive.

In [ ]:
%cd /content/icl-dynamics
!ln -sfn /content/drive/MyDrive/nlp_p1/collapse collapse   # he copy made with the Drive-copy cells
!ln -sfn /content/drive/MyDrive/nlp_p1/collapse_ext collapse_ext
!rm -rf results && ln -sfn /content/drive/MyDrive/nlp_p1/results results
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PY'] = '/content/py310/bin/python'
os.environ['TRAIN_ARGS'] = "--data_file omniglot_resnet18_randomized_order_s0.h5 --mixing_coeffs 1.0 --pt_burstiness 1 --train_context_len 2 --fs_relabel 5 --fs_relabel_split 8 2 0 --exemplar_split 1 4 0 --pe_names fsl_train fsl_val_rl fsl_train_valex fsl_test_class --pe_classes train train train test --pe_exemplars train train val train --pe_fs_relabel_scheme train val train train --pe_burstiness 1 1 1 1 --train_bs 32 --eval_iters 1000 --lr 0.00001 --d_model 64 --class_split 50 1473 100 --raw_name"
!ls -la | grep -E "collapse|results"
os.environ['MPLBACKEND'] = 'Agg'   #Colab's own plot setting breaks matplotlib in the py310 environment


## Step 1: check that everything works 

In [ ]:
!$PY test_interp_heads.py collapse/gen0_pool

## Step 2: generation 0, find the heads (base version)
Writes `results/interp/gen0_base/` in Drive. The summary prints below; the heatmap is shown after it.

In [ ]:
!$PY -m interp.find_heads_gen0 collapse/gen0_pool --out results/interp/gen0_base | head -n 80

In [ ]:
from IPython.display import Image, display
display(Image('results/interp/gen0_base/gen0_head_heatmaps.png', width=650))
display(Image('results/interp/gen0_base/gen0_patterns.png', width=650))
!$PY -c "import numpy as np; d=np.load('results/interp/gen0_base/gen0_scores.npz'); [print(k, chr(10), np.round(d[k],2)) for k in ['prev_token','induction','matching']]"


## Step 3: track across generations
Final checkpoint of every generation only. It checks everything runs and gives the across-generation answer early.

In [ ]:
!$PY -m interp.track_generations --run sample_T2=collapse/gen0_pool:collapse/sample_T2 --run sample_T2_keep10=collapse/gen0_pool:collapse/sample_T2_keep10 --run sample_T1=collapse/gen0_pool:collapse/sample_T1 --run argmax=collapse/gen0_pool:collapse/argmax --gen0-json results/interp/gen0_base/gen0_heads.json --out results/interp/tracking_base

## Step 4: every checkpoint of every generation
About 4,400 checkpoints.

In [ ]:
!$PY -m interp.track_generations --run sample_T2=collapse/gen0_pool:collapse/sample_T2 --run sample_T2_keep10=collapse/gen0_pool:collapse/sample_T2_keep10 --run sample_T1=collapse/gen0_pool:collapse/sample_T1 --run argmax=collapse/gen0_pool:collapse/argmax --gen0-json results/interp/gen0_base/gen0_heads.json --out results/interp/tracking_base --within-training 1

In [ ]:
from IPython.display import Image, display
display(Image('results/interp/tracking_base/tracking.png', width=900))
display(Image('results/interp/tracking_base/within_training.png', width=900))
import csv
print('EARLY WARNING (matching score of the strongest induction head vs each behavioural measure):')
for r in csv.DictReader(open('results/interp/tracking_base/early_warning.csv')):
    if r['circuit_measure'] == 'matching_max':
        print(f"{r['run']:<18}{r['behaviour']:<18} circuit@{r['circuit_first_gen'] or '-':>3}  behaviour@{r['behaviour_first_gen'] or '-':>3}   {r['verdict']}")
print()
print('FORMATION within training (iterations = training sequences seen):')
for r in csv.DictReader(open('results/interp/tracking_base/formation.csv')):
    print(r)

## Step 5: extended version, generation 0


In [ ]:
!$PY -m interp.find_heads_gen0 collapse_ext/gen0_p50 --out results/interp/gen0_ext | head -n 80
from IPython.display import Image, display
display(Image('results/interp/gen0_ext/gen0_head_heatmaps.png', width=650))

## Step 6: extended version, every checkpoint of every generation

In [ ]:
!$PY -m interp.track_generations --run ext_T1=collapse_ext/gen0_p50:collapse_ext/T1 --run ext_argmax=collapse_ext/gen0_p50:collapse_ext/argmax --gen0-json results/interp/gen0_ext/gen0_heads.json --out results/interp/tracking_ext --within-training 1

In [ ]:
from IPython.display import Image, display
display(Image('results/interp/tracking_ext/tracking.png', width=900))
display(Image('results/interp/tracking_ext/within_training.png', width=900))
import csv
for r in csv.DictReader(open('results/interp/tracking_ext/early_warning.csv')):
    if r['circuit_measure'] == 'matching_max':
        print(f"{r['run']:<18}{r['behaviour']:<18} circuit@{r['circuit_first_gen'] or '-':>3}  behaviour@{r['behaviour_first_gen'] or '-':>3}   {r['verdict']}")

## Step 7: robustness of the early-warning call
Reruns only the early-warning test (fast, everything is cached) with a stricter and a looser noise threshold. 

In [ ]:
!$PY -m interp.track_generations --run sample_T2=collapse/gen0_pool:collapse/sample_T2 --run sample_T2_keep10=collapse/gen0_pool:collapse/sample_T2_keep10 --run sample_T1=collapse/gen0_pool:collapse/sample_T1 --run argmax=collapse/gen0_pool:collapse/argmax --gen0-json results/interp/gen0_base/gen0_heads.json --out results/interp/tracking_base_z2 --z 2 > /dev/null && echo z2 done
!$PY -m interp.track_generations --run sample_T2=collapse/gen0_pool:collapse/sample_T2 --run sample_T2_keep10=collapse/gen0_pool:collapse/sample_T2_keep10 --run sample_T1=collapse/gen0_pool:collapse/sample_T1 --run argmax=collapse/gen0_pool:collapse/argmax --gen0-json results/interp/gen0_base/gen0_heads.json --out results/interp/tracking_base_z5 --z 5 > /dev/null && echo z5 done



All results are in **Drive → MyDrive → nlp_p1 → results → interp**. 